In [ ]:
# ========== 导入：网页抓取 + 本地 Ollama（OpenAI 兼容）==========

# 导入标准库 os：读环境变量等（本格后面未必立刻用到，但常与 dotenv 搭配）
import os
# 导入 requests：用 HTTP GET 拉取网页 HTML
import requests
# 从 dotenv 导入 load_dotenv：如需 .env 可在后续格调用（本格只导入）
from dotenv import load_dotenv
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的文档树
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown、display：在笔记本里漂亮渲染摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：后面会把 base_url 指到本地 Ollama
from openai import OpenAI

# 如果运行此单元时出现错误，请转到故障排除笔记本！


In [ ]:
# ========== 客户端：指向本机 Ollama 的 OpenAI 兼容端点 ==========

# base_url 指向 localhost:11434/v1；api_key 对 Ollama 可为任意非空占位（常用 'ollama'）
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
# ========== Website 类：把 URL 抓成「标题 + 正文文本」==========
# 如果还不熟 class，可先对照课程里的「中级 Python」笔记本

# 有些网站会拦无头请求：带上浏览器风格的 User-Agent 请求头（Headers）
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """用给定 url 抓取页面，并用 BeautifulSoup 解析出标题与正文。"""
        # 保存原始地址，便于调试或后续扩展
        self.url = url
        # GET 网页；headers 降低被拒概率
        response = requests.get(url, headers=headers)
        # 用 html.parser 把字节内容建成 DOM 树
        soup = BeautifulSoup(response.content, 'html.parser')
        # 有 <title> 就取字符串，否则用占位英文（保留原字符串）
        self.title = soup.title.string if soup.title else "No title found"
        # 删掉 script/style/img/input 等与摘要无关的节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 抽出可见文本：换行分隔并 strip 空白
        self.text = soup.body.get_text(separator="\n", strip=True)


In [ ]:
# ========== 试抓一个站点：看 title / text 长什么样 ==========

# 实例化 Website；URL 保持原样（可自行改成别的站点再跑）
ed = Website("https://marc-views.vercel.app/")
# 打印页面标题
print(ed.title)
# 打印清理后的正文（可能很长）
print(ed.text)


In [ ]:
# ========== system prompt：告诉模型「怎么总结网站」==========
# 可实验：把最后一句改成别的语言要求；但改 prompt 会改变模型行为，本教学注释不改字符串

# 多行用反斜杠续行；内容是发给模型的 system 指令，保留英文
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."


In [ ]:
# ========== user_prompt_for：根据 Website 对象拼用户侧提示 ==========

def user_prompt_for(website):
    # 先写标题行（f-string 嵌入 website.title）
    user_prompt = f"You are looking at a website titled {website.title}"
    # 再追加「请用 markdown 做短摘要」的说明（英文 prompt 原样保留）
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    # 最后把网页正文贴进去，模型才能「看见」页面内容
    user_prompt += website.text
    # 返回完整 user 字符串，供 messages 使用
    return user_prompt


In [ ]:
# 预览拼好的 user prompt（可能很长）：确认标题与正文都进去了
print(user_prompt_for(ed))


In [ ]:
# ========== 小例子：先熟悉 messages 列表结构（与网页摘要无关的热身）==========

# Chat Completions 的标准格式：system 定性格，user 提问
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]


In [ ]:
# ========== 热身调用：用上面的 messages 打本地 llama3.2 ==========

# model 名必须与本机 ollama list 一致；走的是本格之前创建的 openai 客户端（其实是 Ollama）
response = openai.chat.completions.create(model="llama3.2", messages=messages)
# 打印助手回复正文
print(response.choices[0].message.content)


In [ ]:
# ========== messages_for：把 system_prompt + user_prompt_for 收成 messages ==========

# 返回的结构和上面热身例子相同：[{system...}, {user...}]
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [ ]:
# 试着看一眼 messages_for(ed) 的结构；也可换别的 Website 再试

messages_for(ed)


In [ ]:
# ========== summarize：URL → 抓取 → 调本地模型 → 返回摘要文本 ==========

def summarize(url):
    # 先把 URL 变成 Website（标题+正文）
    website = Website(url)
    # 再调用 Chat Completions；messages 由 messages_for 组装
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    # 只返回助手文本，方便上层再 display
    return response.choices[0].message.content


In [ ]:
# 直接调用 summarize：输出是原始字符串（笔记本里可能显示为 Out[]）
summarize("https://marc-views.vercel.app/")


In [ ]:
# ========== display_summary：先 summarize，再以 Markdown 渲染 ==========

def display_summary(url):
    # 拿到模型生成的 markdown 字符串
    summary = summarize(url)
    # 在 Jupyter 里漂亮显示（而不是纯文本转义）
    display(Markdown(summary))


In [ ]:
# 端到端演示：抓页 → 本地 Llama 摘要 → Markdown 展示
display_summary("https://marc-views.vercel.app/")


In [ ]:
# ========== 小练习：根据「邮件正文」让模型建议主题行（Subject Line）==========

# 第 1 步：创建提示（system / user 字符串都是发给模型的指令，保留英文原样）

system_prompt = "You are a helpful assitant, mindful about the user which seems not so good at speaking english, because its his second language. You are trying to suggest an appropriate and impactful subject line for the email content provided by the User!"
user_prompt = """
    Hii I am rava, I am writing this email so I ask you if you remember that you told me to come to interview tomorrow only if i have AI experience already, the position is AI engineer, but i have been only knowing pyhton, working into full stack position. Forgive my english if you think i am not able to speak, then i should tell you, my english might broke, but i will not. I am studying, working, never lacking the courage and attitude to show that knowing less is not a flaw, wanting not to learn is, so I am telling you through this email, you will be happy to offering me this chance.
"""

# 第 2 步：创建消息列表（system + user）；行尾 # fill this in 是原作者提示，保留

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}] # fill this in

# 第三步：调用本地 Ollama（变量名仍叫 openai，base_url 已指向本机）

response = openai.chat.completions.create(
    model= "llama3.2",
    messages=messages
)

# 第四步：打印结果（主题行建议）

print(response.choices[0].message.content)
